In [1]:
from fractions import Fraction

class MomentDistributionSolver:
    def __init__(self):
        self.members = {}
        self.ends = []
        self.M = {}
        self.end_to_member = {}
        self.far_end = {}
        self.joint_ends = {}
        self.rows = []

    def add_member(
        self,
        name,
        i,
        j,
        L,
        I=1,
        fem_ij=0,
        fem_ji=0,
        cof_ij=Fraction(1, 2),
        cof_ji=Fraction(1, 2),
    ):
        """
        Add a member i-j.

        name: member name, e.g. "AB"
        i, j: joint names
        L: member length
        I: moment of inertia, E cancels if same material
        fem_ij: fixed-end moment at i end
        fem_ji: fixed-end moment at j end
        cof_ij: carry-over factor from i end to j end
        cof_ji: carry-over factor from j end to i end

        Sign convention:
        member-end clockwise moment is positive.
        """

        L = Fraction(str(L))
        I = Fraction(str(I))
        fem_ij = Fraction(str(fem_ij))
        fem_ji = Fraction(str(fem_ji))

        end_ij = f"{i}{j}"
        end_ji = f"{j}{i}"

        self.members[name] = {
            "i": i,
            "j": j,
            "L": L,
            "I": I,
            "K": Fraction(4) * I / L,
            "end_ij": end_ij,
            "end_ji": end_ji,
            "cof_ij": Fraction(cof_ij),
            "cof_ji": Fraction(cof_ji),
        }

        self.ends += [end_ij, end_ji]

        self.M[end_ij] = fem_ij
        self.M[end_ji] = fem_ji

        self.end_to_member[end_ij] = name
        self.end_to_member[end_ji] = name

        self.far_end[end_ij] = end_ji
        self.far_end[end_ji] = end_ij

        if i not in self.joint_ends:
            self.joint_ends[i] = []
        if j not in self.joint_ends:
            self.joint_ends[j] = []

        self.joint_ends[i].append(end_ij)
        self.joint_ends[j].append(end_ji)

    def get_stiffness(self, end):
        member_name = self.end_to_member[end]
        return self.members[member_name]["K"]

    def get_carry_over(self, end):
        member_name = self.end_to_member[end]
        member = self.members[member_name]

        if end == member["end_ij"]:
            return member["cof_ij"]
        else:
            return member["cof_ji"]

    def distribution_factors(self, joint):
        ends = self.joint_ends[joint]
        total_K = sum(self.get_stiffness(e) for e in ends)

        return {e: self.get_stiffness(e) / total_K for e in ends}

    def balance_joint(self, joint, label=None):
        """
        Balance one joint once.
        """

        ends = self.joint_ends[joint]
        DF = self.distribution_factors(joint)

        unbalanced = sum(self.M[e] for e in ends)
        correction = -unbalanced

        inc = {e: Fraction(0) for e in self.ends}

        for e in ends:
            distributed = correction * DF[e]
            inc[e] += distributed

            far = self.far_end[e]
            cof = self.get_carry_over(e)
            inc[far] += distributed * cof

        for e in self.ends:
            self.M[e] += inc[e]

        if label is None:
            label = f"Balance {joint}"

        self.rows.append((label, inc.copy()))

    def solve(self, balance_sequence, cycles=10):
        """
        balance_sequence:
        e.g. ["C", "B"] for textbook problem
        e.g. ["A'", "B'", "C'", "D'"] for Royce Hall
        """

        self.rows = []
        self.rows.append(("FEM", self.M.copy()))

        for c in range(1, cycles + 1):
            for joint in balance_sequence:
                self.balance_joint(joint, f"Balance {joint} {c}")

        return self.M

    def print_distribution_factors(self, joints):
        print("Distribution Factors")
        print("-" * 60)
        for joint in joints:
            DF = self.distribution_factors(joint)
            print(joint, {e: fmt_frac(v) for e, v in DF.items()})
        print()

    def print_table(self, decimals=1):
        print("Moment Distribution Table")
        print("-" * 120)

        print(f"{'Step':<18}", end="")
        for e in self.ends:
            print(f"{e:>12}", end="")
        print()

        for name, row in self.rows:
            print(f"{name:<18}", end="")
            for e in self.ends:
                print(f"{fmt_dec(row.get(e, 0), decimals):>12}", end="")
            print()

        print("-" * 120)
        print(f"{'Total':<18}", end="")
        for e in self.ends:
            print(f"{fmt_dec(self.M[e], decimals):>12}", end="")
        print()
        print()

    def print_final(self):
        print("Final End Moments")
        print("-" * 60)
        for e in self.ends:
            print(f"{e:>8} = {fmt_frac(self.M[e])}")
        print()


def fmt_frac(x):
    x = Fraction(x)
    if x == 0:
        return "0"
    if x.denominator == 1:
        return str(x.numerator)
    return f"{x.numerator}/{x.denominator}"


def fmt_dec(x, decimals=1):
    x = float(x)
    if abs(x) < 10 ** (-(decimals + 1)):
        x = 0
    return f"{x:.{decimals}f}"

In [6]:
#選用課本的例題，用來驗證程式碼的正確性

def solve_textbook_problem(): 

    solver = MomentDistributionSolver()

    w = Fraction(6000)
    L_BC = Fraction(4)

    FEM = w * L_BC**2 / 12

    # AB has no load.
    solver.add_member(
        name="AB",
        i="A",
        j="B",
        L=3,
        I=120,
        fem_ij=0,
        fem_ji=0,
    )

    # BC has UDL.
    # Fixed-end moment:
    # M_BC = -wL^2/12
    # M_CB = +wL^2/12
    solver.add_member(
        name="BC",
        i="B",
        j="C",
        L=4,
        I=240,
        fem_ij=-FEM,
        fem_ji=+FEM,
    )

    # Only balance C and B.
    # A is fixed, so not balanced.
    # C is released, so we balance it to make M_CB -> 0.
    balance_sequence = ["C", "B"]

    solver.solve(balance_sequence, cycles=8)

    solver.print_distribution_factors(["B", "C"])
    solver.print_table(decimals=1)
    solver.print_final()

    return solver


textbook_solver = solve_textbook_problem()

Distribution Factors
------------------------------------------------------------
B {'BA': '2/5', 'BC': '3/5'}
C {'CB': '1'}

Moment Distribution Table
------------------------------------------------------------------------------------------------------------------------
Step                        AB          BA          BC          CB
FEM                        0.0         0.0     -8000.0      8000.0
Balance C 1                0.0         0.0     -4000.0     -8000.0
Balance B 1             2400.0      4800.0      7200.0      3600.0
Balance C 2                0.0         0.0     -1800.0     -3600.0
Balance B 2              360.0       720.0      1080.0       540.0
Balance C 3                0.0         0.0      -270.0      -540.0
Balance B 3               54.0       108.0       162.0        81.0
Balance C 4                0.0         0.0       -40.5       -81.0
Balance B 4                8.1        16.2        24.3        12.2
Balance C 5                0.0         0.0        -6.1   

In [8]:
from fractions import Fraction

class MomentDistributionSolver:
    def __init__(self):
        self.members = {}
        self.ends = []
        self.M = {}
        self.end_to_member = {}
        self.far_end = {}
        self.joint_ends = {}
        self.rows = []

    def add_member(self, name, i, j, L, I=1, fem_ij=0, fem_ji=0):
        L = Fraction(str(L))
        I = Fraction(str(I))
        fem_ij = Fraction(str(fem_ij))
        fem_ji = Fraction(str(fem_ji))

        end_ij = f"{i}{j}"
        end_ji = f"{j}{i}"

        self.members[name] = {
            "i": i,
            "j": j,
            "L": L,
            "I": I,
            "K": Fraction(4) * I / L,
            "end_ij": end_ij,
            "end_ji": end_ji,
        }

        self.ends += [end_ij, end_ji]

        self.M[end_ij] = fem_ij
        self.M[end_ji] = fem_ji

        self.end_to_member[end_ij] = name
        self.end_to_member[end_ji] = name

        self.far_end[end_ij] = end_ji
        self.far_end[end_ji] = end_ij

        self.joint_ends.setdefault(i, []).append(end_ij)
        self.joint_ends.setdefault(j, []).append(end_ji)

    def get_stiffness(self, end):
        return self.members[self.end_to_member[end]]["K"]

    def distribution_factors(self, joint):
        ends = self.joint_ends[joint]
        total_K = sum(self.get_stiffness(e) for e in ends)
        return {e: self.get_stiffness(e) / total_K for e in ends}

    def balance_joint(self, joint, label=None):
        ends = self.joint_ends[joint]
        DF = self.distribution_factors(joint)

        unbalanced = sum(self.M[e] for e in ends)
        correction = -unbalanced

        inc = {e: Fraction(0) for e in self.ends}

        for e in ends:
            distributed = correction * DF[e]
            inc[e] += distributed

            far = self.far_end[e]
            inc[far] += distributed / 2

        for e in self.ends:
            self.M[e] += inc[e]

        self.rows.append((label or f"Balance {joint}", inc.copy()))

    def solve(self, balance_sequence, cycles=12):
        self.rows = []
        self.rows.append(("FEM", self.M.copy()))

        for c in range(1, cycles + 1):
            for joint in balance_sequence:
                self.balance_joint(joint, f"Balance {joint} {c}")

        return self.M


def fmt_frac(x):
    x = Fraction(x).limit_denominator()
    if x == 0:
        return "0"
    if x.denominator == 1:
        return str(x.numerator)
    return f"{x.numerator}/{x.denominator}"


def fmt_dec(x, decimals=5):
    return f"{float(x):.{decimals}f}"


def print_md_table(solver, order, decimals=5):
    print(f"{'Step':<18}", end="")
    for e in order:
        print(f"{e:>12}", end="")
    print()

    print("-" * (18 + 12 * len(order)))

    for name, row in solver.rows:
        print(f"{name:<18}", end="")
        for e in order:
            print(f"{fmt_dec(row.get(e, 0), decimals):>12}", end="")
        print()

    print("-" * (18 + 12 * len(order)))

    print(f"{'Total':<18}", end="")
    for e in order:
        print(f"{fmt_dec(solver.M[e], decimals):>12}", end="")
    print()


def solve_royce_hall():
    solver = MomentDistributionSolver()

    # Geometry
    L = Fraction(7, 1)      # column height = 7 m
    l = Fraction(7, 2)      # beam span = 3.5 m
    P = Fraction(1, 1)      # use P = 1, so all results are coefficients of P

    # Fixed-end moment for beam with midspan point load P
    FEM = P * l / 8

    # Columns
    solver.add_member("AA'", "A", "A'", L, I=1)
    solver.add_member("BB'", "B", "B'", L, I=1)
    solver.add_member("CC'", "C", "C'", L, I=1)
    solver.add_member("DD'", "D", "D'", L, I=1)

    # Beams
    solver.add_member("A'B'", "A'", "B'", l, I=1, fem_ij=-FEM, fem_ji=+FEM)
    solver.add_member("B'C'", "B'", "C'", l, I=1, fem_ij=-FEM, fem_ji=+FEM)
    solver.add_member("C'D'", "C'", "D'", l, I=1, fem_ij=-FEM, fem_ji=+FEM)

    # Only top joints are balanced.
    # Bottom A, B, C, D are fixed supports, so they only receive carry-over.
    balance_sequence = ["A'", "B'", "C'", "D'"]

    solver.solve(balance_sequence, cycles=16)

    return solver


royce_solver = solve_royce_hall()

ppt_order = [
    "AA'", "A'A", "A'B'",
    "B'A'", "B'B", "BB'", "B'C'",
    "C'B'", "C'C", "CC'", "C'D'",
    "D'C'", "D'D", "DD'"
]

print("Distribution Factors")
print("-" * 50)
for joint in ["A'", "B'", "C'", "D'"]:
    DF = royce_solver.distribution_factors(joint)
    print(joint, {e: fmt_frac(v) for e, v in DF.items()})

print("\nMoment Distribution Table")
print_md_table(royce_solver, ppt_order, decimals=5)

print("\nFinal Moments in terms of P")
print("-" * 50)
for e in ppt_order:
    print(f"{e:>6} = {fmt_frac(royce_solver.M[e])} P")

Distribution Factors
--------------------------------------------------
A' {"A'A": '1/3', "A'B'": '2/3'}
B' {"B'B": '1/5', "B'A'": '2/5', "B'C'": '2/5'}
C' {"C'C": '1/5', "C'B'": '2/5', "C'D'": '2/5'}
D' {"D'D": '1/3', "D'C'": '2/3'}

Moment Distribution Table
Step                       AA'         A'A        A'B'        B'A'         B'B         BB'        B'C'        C'B'         C'C         CC'        C'D'        D'C'         D'D         DD'
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
FEM                    0.00000     0.00000    -0.43750     0.43750     0.00000     0.00000    -0.43750     0.43750     0.00000     0.00000    -0.43750     0.43750     0.00000     0.00000
Balance A' 1           0.07292     0.14583     0.29167     0.14583     0.00000     0.00000     0.00000     0.00000     0.00000     0.00000     0.00000     0.00000     0.00000    